# RAG 第 13 课：LLM-as-a-Judge

前面我们已经学了很多评估方式：
- `RetrievalHit@k`
- `Exact Match`
- `Token F1`
- 失败样本分析

这些都很重要，但真实项目里还会遇到一个问题：

- 有些答案不是完全字符串匹配
- 但它其实是对的
- 或者它看起来像对的，但其实偷偷加了上下文里没有的信息

所以这节课我们要学一个非常实用的思路：

**让 LLM 来做评委（Judge）。**

也就是：
- 给它问题
- 给它上下文
- 给它模型答案
- 再让它判断答案是否正确、是否被上下文支持

In [ ]:
# 先清理 GPU 缓存
import gc

try:
    import torch
    if torch.cuda.is_available():
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        print('GPU cache cleared.')
    else:
        print('CUDA not available, skip GPU cache clear.')
except Exception as e:
    print(f'Torch not available, skip GPU cache clear. ({e})')

## 1. 连接本地模型

这一课我们继续使用你本地的 LM Studio OpenAI 兼容接口。
默认优先使用当前可用的：
- `qwen3.5-35b-a3b` 作为 judge / answer model
- `text-embedding-nomic-embed-text-v1.5` 作为 embedding model

In [ ]:
import json
import re
from collections import Counter
from typing import List

import numpy as np
from datasets import load_dataset
from openai import OpenAI

BASE_URL = 'http://10.0.0.63:1234/v1'
API_KEY = 'lm-studio'

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)
model_ids = [m.id for m in client.models.list().data]
print('available models:', model_ids)

preferred_chat_models = ['qwen3.5-35b-a3b', 'qwen3.5-27b']
chat_model = next((m for m in preferred_chat_models if m in model_ids), None)
embedding_model = next((m for m in model_ids if 'embed' in m.lower() or 'embedding' in m.lower()), None)

print('chat_model      =', chat_model)
print('embedding_model =', embedding_model)

if chat_model is None:
    raise RuntimeError('没有找到可用的 chat model。')
if embedding_model is None:
    raise RuntimeError('没有找到可用的 embedding model。')

## 2. 加载真实数据集并构建最小 RAG

为了聚焦 judge 逻辑，我们继续沿用 `squad`，并保留一个最小真实 RAG：
- embedding 检索
- top-k 上下文
- 本地 LLM 回答

In [ ]:
raw_ds = load_dataset('squad', split='validation[:120]')

context_to_doc_id = {}
documents = []
eval_rows = []

for item in raw_ds:
    context = item['context'].strip()
    if context not in context_to_doc_id:
        doc_id = len(documents)
        context_to_doc_id[context] = doc_id
        documents.append({'doc_id': doc_id, 'text': context, 'title': item['title']})
    else:
        doc_id = context_to_doc_id[context]

    if item['answers']['text']:
        eval_rows.append({
            'question': item['question'].strip(),
            'gold_doc_id': doc_id,
            'reference_answer': item['answers']['text'][0].strip(),
            'title': item['title'],
        })

print('num_documents =', len(documents))
print('num_eval_rows =', len(eval_rows))

In [ ]:
def get_embeddings(texts: List[str], batch_size: int = 16) -> np.ndarray:
    all_vectors = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        response = client.embeddings.create(model=embedding_model, input=batch)
        batch_vectors = [np.array(item.embedding, dtype=np.float32) for item in response.data]
        all_vectors.extend(batch_vectors)
    return np.vstack(all_vectors)


def l2_normalize(matrix: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    norms = np.clip(norms, 1e-12, None)
    return matrix / norms

In [ ]:
doc_texts = [doc['text'] for doc in documents]
doc_embeddings = l2_normalize(get_embeddings(doc_texts, batch_size=16))
print('doc_embeddings shape =', doc_embeddings.shape)

In [ ]:
def retrieve(question: str, top_k: int = 3):
    q_emb = l2_normalize(get_embeddings([question], batch_size=1))[0]
    scores = doc_embeddings @ q_emb
    top_indices = np.argsort(scores)[::-1][:top_k]
    rows = []
    for idx in top_indices:
        rows.append({
            'doc_id': int(documents[idx]['doc_id']),
            'title': documents[idx]['title'],
            'score': float(scores[idx]),
            'text': documents[idx]['text'],
        })
    return rows


def build_context_block(retrieved_docs):
    parts = []
    for i, doc in enumerate(retrieved_docs, start=1):
        parts.append(f'[Document {i}] title: {doc["title"]}\n{doc["text"]}')
    return '\n\n'.join(parts)


def answer_with_llm(question: str, retrieved_docs):
    system_prompt = (
        'You are a careful question-answering assistant. '
        'Answer only from the provided context. '
        'If the answer is not supported by the context, reply with NOT_FOUND. '
        'Keep the answer short.'
    )
    user_prompt = (
        f'Question: {question}\n\n'
        f'Context:\n{build_context_block(retrieved_docs)}\n\n'
        'Return only the answer.'
    )
    response = client.chat.completions.create(
        model=chat_model,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt},
        ],
        temperature=0,
    )
    return response.choices[0].message.content.strip()

## 3. 传统字符串指标仍然保留

Judge 不是用来完全替代传统指标，而是做一个补充。
所以这一课我们会同时保留：
- `Exact Match`
- `Token F1`
- `LLM Judge`

In [ ]:
def normalize_token(token):
    token = token.lower()
    token = re.sub(r'[^a-z0-9]+', '', token)
    return token


def tokenize(text):
    return [t for t in (normalize_token(x) for x in text.split()) if t]


def normalize_text(text):
    return ' '.join(tokenize(text))


def exact_match(prediction, reference):
    return 1.0 if normalize_text(prediction) == normalize_text(reference) else 0.0


def token_f1(prediction, reference):
    pred_tokens = tokenize(prediction)
    ref_tokens = tokenize(reference)
    pred_counter = Counter(pred_tokens)
    ref_counter = Counter(ref_tokens)
    overlap = sum((pred_counter & ref_counter).values())
    if overlap == 0:
        return 0.0
    precision = overlap / len(pred_tokens)
    recall = overlap / len(ref_tokens)
    return 2 * precision * recall / (precision + recall)

## 4. 定义 Judge 提示词

这里我们让同一个本地模型扮演“评委”。
它要看三样东西：
- 问题
- 上下文
- 模型答案

然后输出一个结构化判断：
- `correct`：答案是否正确
- `grounded`：答案是否被上下文支持
- `score`：一个 0 到 1 的总分
- `reason`：简短解释

In [ ]:
def judge_answer(question: str, context_docs, predicted_answer: str, reference_answer: str):
    context_block = build_context_block(context_docs)
    system_prompt = (
        'You are an evaluation judge for a retrieval-augmented QA system. '
        'Evaluate whether the predicted answer is correct and supported by the provided context. '
        'Return strict JSON only.'
    )
    user_prompt = (
        f'Question: {question}\n\n'
        f'Context:\n{context_block}\n\n'
        f'Predicted answer: {predicted_answer}\n'
        f'Reference answer: {reference_answer}\n\n'
        'Return a JSON object with keys: correct, grounded, score, reason. '
        'correct and grounded must be booleans. '
        'score must be a float between 0 and 1. '
        'reason must be a short explanation.'
    )

    response = client.chat.completions.create(
        model=chat_model,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt},
        ],
        temperature=0,
    )

    raw = response.choices[0].message.content.strip()
    match = re.search(r'\{.*\}', raw, flags=re.S)
    if not match:
        raise ValueError(f'Judge 输出中没有找到 JSON: {raw}')
    parsed = json.loads(match.group(0))
    return parsed, raw

## 5. 先看单条样本

这一格会很直观地展示 Judge 是怎么工作的。

In [ ]:
sample = eval_rows[0]
retrieved = retrieve(sample['question'], top_k=3)
pred_answer = answer_with_llm(sample['question'], retrieved)
judge_result, judge_raw = judge_answer(sample['question'], retrieved, pred_answer, sample['reference_answer'])

print('question:')
print(sample['question'])
print('\nreference_answer:')
print(sample['reference_answer'])
print('\npred_answer:')
print(pred_answer)
print('\njudge raw:')
print(judge_raw)
print('\njudge parsed:')
print(judge_result)

## 6. 小规模批量评估

为了控制调用次数，这里我们先跑前 5 条。
这样已经足够看出 Judge 指标和字符串指标之间的差异。

In [ ]:
eval_subset = eval_rows[:5]
results = []

for i, row in enumerate(eval_subset, start=1):
    print(f'processing {i}/{len(eval_subset)}: {row["question"]}')
    retrieved = retrieve(row['question'], top_k=3)
    pred_answer = answer_with_llm(row['question'], retrieved)
    judge_result, _ = judge_answer(row['question'], retrieved, pred_answer, row['reference_answer'])

    results.append({
        'question': row['question'],
        'reference_answer': row['reference_answer'],
        'pred_answer': pred_answer,
        'exact_match': exact_match(pred_answer, row['reference_answer']),
        'token_f1': token_f1(pred_answer, row['reference_answer']),
        'judge_correct': 1.0 if judge_result.get('correct', False) else 0.0,
        'judge_grounded': 1.0 if judge_result.get('grounded', False) else 0.0,
        'judge_score': float(judge_result.get('score', 0.0)),
        'judge_reason': judge_result.get('reason', ''),
    })

summary = {
    'ExactMatch': float(np.mean([x['exact_match'] for x in results])),
    'TokenF1': float(np.mean([x['token_f1'] for x in results])),
    'JudgeCorrectRate': float(np.mean([x['judge_correct'] for x in results])),
    'JudgeGroundedRate': float(np.mean([x['judge_grounded'] for x in results])),
    'JudgeScore': float(np.mean([x['judge_score'] for x in results])),
}

print('\nsummary =', summary)

## 7. 看逐条结果

这是这节课最有价值的部分。

你会看到一些非常典型的情况：
- `Exact Match = 0`，但 Judge 认为答案其实是对的
- `Token F1` 不低，但 Judge 觉得答案不够 grounded
- Judge 的解释能帮助你更快定位问题

In [ ]:
for row in results:
    print('question:', row['question'])
    print('reference_answer:', row['reference_answer'])
    print('pred_answer:', row['pred_answer'])
    print('ExactMatch:', row['exact_match'], '| TokenF1:', round(row['token_f1'], 3))
    print('JudgeCorrect:', row['judge_correct'], '| JudgeGrounded:', row['judge_grounded'], '| JudgeScore:', round(row['judge_score'], 3))
    print('JudgeReason:', row['judge_reason'])
    print('-' * 100)

## 8. 这节课最想让你建立的直觉

Judge 不是银弹，但它很有用，因为它能捕捉到一些字符串指标抓不到的东西。

比如：
- `Denver Broncos` vs `The Denver Broncos`
- `Santa Clara, California` vs `Levi's Stadium in Santa Clara, California`

这些情况下：
- `Exact Match` 往往太严格
- `Token F1` 虽然有帮助，但仍然不理解语义
- `LLM Judge` 则更接近“人类评审”的感觉

当然它也有代价：
- 更慢
- 更贵
- Judge 自己也可能不稳定

所以真实项目里，常见做法是：
- 基础指标继续保留
- 再加一层 LLM Judge 作为补充

## 9. 本课小结

这节课你要带走的核心是：

1. `LLM-as-a-Judge` 是 RAG 评估里非常实用的一条路线。
2. Judge 能帮助你评估 correctness 和 groundedness。
3. 它适合作为 `Exact Match / Token F1` 的补充，而不是完全替代。
4. Judge 的解释文本对错误分析很有价值。

下一课最自然的衔接就是：
**真实版 Hybrid Search**，也就是把 lexical 检索和 embedding 检索真正融合起来。**